In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
import os
from scipy.sparse.linalg import eigs

In [2]:
def compose_weight_tensor(BASE_PATH, arg1, arg2, arg3 ):
    layers = [1,2]
    gates = ['forget', 'input', 'output', 'cell']
    weight_types = ['ih', 'hh']
    checkpoints = {}
    for layer in layers:
        for gate in gates:
            for weight_type in weight_types:
                check = []
                check.append(torch.load(f'{BASE_PATH}/layer{layer}_{gate}_gate_{weight_type}_{arg1}.pt'))
                check.append(torch.load(f'{BASE_PATH}/layer{layer}_{gate}_gate_{weight_type}_{arg2}.pt'))
                check.append(torch.load(f'{BASE_PATH}/layer{layer}_{gate}_gate_{weight_type}_{arg3}.pt'))
                check = torch.cat(check, dim=0)
                checkpoints[f'layer{layer}_{gate}_gate_{weight_type}'] = check
    return checkpoints

In [3]:
BASE_PATH = '/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam_full_check/weights'
checkpoints = compose_weight_tensor(
    BASE_PATH,
    'first100batches',
    'every100batches',
    'ep'
)

In [4]:
xvals = []
for i in range(0,101,1):
    xvals.append(i/9269)
for i in range(200,9300,100):
    xvals.append(i/9269)
for i in range(1,41,1):
    xvals.append(i)
    
x_first_ep = []
for i in range(0,101,1):
    x_first_ep.append(i/9269)
for i in range(200,9300,100):
    x_first_ep.append(i/9269)
    
x_100_first_batches = []
for i in range(0,101,1):
    x_100_first_batches.append(i/9269)
x_ep =[]
for i in range(1,41,1):
    x_ep.append(i)


In [5]:
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_instant_cosine_similarity(checkpoints, xvals, check_name, title):
    # Compute cosine similarities between adjacent matrices
    cos_sims = []
    for t in range(len(xvals) - 1):
        w1 = checkpoints[check_name][t]
        w2 = checkpoints[check_name][t + 1]
        cos_sim = torch.nn.functional.cosine_similarity(w1.flatten(), w2.flatten(), dim=0).item()
        cos_sims.append(cos_sim)

    # Split data by regime
    reg1_end = 100
    reg2_end = reg1_end + (9300 - 200) // 100  # 91 additional coarse batches
    reg1_indices = list(range(1, reg1_end + 1))
    reg2_indices = list(range(reg1_end + 1, reg2_end + 1))
    reg3_indices = list(range(reg2_end + 1, len(xvals)))

    # Create subplots
    fig = make_subplots(rows=3, cols=1, shared_yaxes=True,
                        subplot_titles=("Fine Batch-Level Checkpoints",
                                        "Coarse Batch-Level Checkpoints",
                                        "Epoch-Level Checkpoints"))

    # Helper to add traces
    def add_trace(indices, row, name):
        hover_labels = []
        for idx in indices:
            x = xvals[idx]
            if x < 1:
                batch_num = int(round(x * 9269))
                label = f"Batch {batch_num} of Epoch 1"
            else:
                label = f"Epoch {int(x)}"
            hover_labels.append(label)

        fig.add_trace(go.Scatter(
            x=indices,
            y=[cos_sims[i - 1] for i in indices],
            mode='lines+markers',
            name=name,
            customdata=hover_labels,
            hovertemplate='CosSim(Wₜ, Wₜ₊₁): %{y:.4f}<br>%{customdata}<extra></extra>'
        ), row=row, col=1)

    # Add traces for each regime
    add_trace(reg1_indices, row=1, name='Fine Batch')
    add_trace(reg2_indices, row=2, name='Coarse Batch')
    add_trace(reg3_indices, row=3, name='Epoch')

    # Layout adjustments
    fig.update_layout(height=900, width=900,
                      title_text=f"Cosine Similarity of Weights per Regime for {title}",
                      showlegend=False)

    fig.update_xaxes(title_text="Checkpoint Index", row=3)
    fig.update_yaxes(title_text="Cosine Similarity", row=2)

    fig.show()


In [ ]:
layers = [1,2]
gates = ['forget', 'cell', 'input', 'output']
weight_types = ['ih', 'hh']
for layer in layers:
    for gate in gates:
        for weight_type in weight_types:
            plot_instant_cosine_similarity(checkpoints, xvals, f'layer{layer}_{gate}_gate_{weight_type}', f'layer{layer}_{gate}_gate_{weight_type}')

In [7]:
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_cosine_similarity_to_final_subplots(checkpoints, xvals, check_name, title):
    final_weights = checkpoints[check_name][-1].flatten()

    # Compute cosine similarity to final weights
    cos_sims = [
        torch.nn.functional.cosine_similarity(checkpoints[check_name][t].flatten(), final_weights, dim=0).item()
        for t in range(len(xvals))
    ]

    # Define regimes
    reg1_end = 100
    reg2_end = reg1_end + (9300 - 200) // 100  # 91 coarse batches
    reg1_indices = list(range(0, reg1_end))
    reg2_indices = list(range(reg1_end, reg2_end))
    reg3_indices = list(range(reg2_end, len(xvals)))

    # Subplots
    fig = make_subplots(rows=3, cols=1, shared_yaxes=True,
                        subplot_titles=("Fine Batch-Level Checkpoints",
                                        "Coarse Batch-Level Checkpoints",
                                        "Epoch-Level Checkpoints"))

    def add_trace(indices, row, name):
        hover_labels = []
        for idx in indices:
            x = xvals[idx]
            if x < 1:
                batch_num = int(round(x * 9269))
                label = f"Batch {batch_num} of Epoch 1"
            else:
                label = f"Epoch {int(x)}"
            hover_labels.append(label)

        fig.add_trace(go.Scatter(
            x=indices,
            y=[cos_sims[i] for i in indices],
            mode='lines+markers',
            name=name,
            customdata=hover_labels,
            hovertemplate='CosSim(Wₜ, W_final): %{y:.4f}<br>%{customdata}<extra></extra>'
        ), row=row, col=1)

    add_trace(reg1_indices, row=1, name='Fine Batch')
    add_trace(reg2_indices, row=2, name='Coarse Batch')
    add_trace(reg3_indices, row=3, name='Epoch')

    fig.update_layout(height=900, width=900,
                      title_text=f"Cosine Similarity to Final Weights Across Checkpointing Regimes for {title}",
                      showlegend=False)

    fig.update_xaxes(title_text="Checkpoint Index", row=3)
    fig.update_yaxes(title_text="CosSim(Wₜ, W_final)", row=2)

    fig.show()


In [ ]:
layers = [1,2]
gates = ['forget', 'cell', 'input', 'output']
weight_types = ['ih', 'hh']
for layer in layers:
    for gate in gates:
        for weight_type in weight_types:
            plot_cosine_similarity_to_final_subplots(checkpoints, xvals, f'layer{layer}_{gate}_gate_{weight_type}', f'layer{layer}_{gate}_gate_{weight_type}')
